# 01. 데이터 전처리 (Preprocessing)
## 서울 성동구 요식 가맹점 조기 경보 시스템 | 빅콘테스트 2025

> **목적**: 원본 3개 데이터셋(점포 마스터, 월별 매출, 월별 고객)을 로드·정제하여 분석용 패널 데이터 생성
> **관측기간**: 2023-01 ~ 2024-12 (24개월)
> **대상 지역**: 서울 성동구

### 데이터셋 개요

| 파일 | 내용 | 주요 컬럼 |
|---|---|---|
| `big_data_set1_f.csv` | 점포 마스터 | ENCODED_MCT, 개업일, 폐업일, 업종, 상권 |
| `big_data_set2_f.csv` | 월별 매출 | 매출액·이용건수·평균객단가 구간 |
| `big_data_set3_f.csv` | 월별 고객 | 재방문율, 유동인구 이용률, 거주고객 비율 |

### 버킷 컬럼 인코딩 규칙
- 버킷값 `1` = 상위 10% 이하 (최상위), `6` = 하위 10% 이하 (최하위)
- EWS에서는 **값이 클수록 위험** 방향으로 정렬 (`f_` 피처 생성 시 반전 처리)


In [ ]:
# ================================================================
# STEP 1: 환경 설정 및 원본 데이터 로드
# ================================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── 경로 설정 ──────────────────────────────────────────────
DATA_DIR  = '../data/'        # 원본 데이터 폴더 (gitignore 처리)
OUT_DIR   = '../outputs/'     # 중간 산출물 저장

OBS_START = pd.Timestamp('2023-01-01')
OBS_END   = pd.Timestamp('2024-12-31')
REGION    = '성동구'
SENTINEL  = -999999.9        # 결측 대체 sentinel 값

# ── 원본 데이터 로드 ───────────────────────────────────────
print(" 원본 데이터 로드 중...")

df1 = pd.read_csv(f'{DATA_DIR}big_data_set1_f.csv', encoding='cp949')
df2 = pd.read_csv(f'{DATA_DIR}big_data_set2_f.csv', encoding='cp949')
df3 = pd.read_csv(f'{DATA_DIR}big_data_set3_f.csv', encoding='cp949')

print(f"  dataset1 (점포 마스터): {df1.shape[0]:,}행 × {df1.shape[1]}열")
print(f"  dataset2 (월별 매출):   {df2.shape[0]:,}행 × {df2.shape[1]}열")
print(f"  dataset3 (월별 고객):   {df3.shape[0]:,}행 × {df3.shape[1]}열")


---
## STEP 2. 점포 마스터 구축 (dataset1)
성동구 점포만 필터링하고, 개업/폐업일을 datetime으로 변환한 뒤 관측기간 내 폐업 레이블 생성.


In [ ]:
# ================================================================
# STEP 2: 점포 마스터 구축 (dataset1)
# ================================================================

# 성동구 필터링
master = df1[df1['MCT_BSE_AR'].str.contains(REGION, na=False)].copy()
print(f" 성동구 점포 수: {len(master):,}개")

# 날짜 파싱
master['ARE_D_dt'] = pd.to_datetime(
    master['ARE_D'].astype(str), format='%Y%m%d', errors='coerce'
)
master['MCT_ME_D_dt'] = master['MCT_ME_D'].apply(
    lambda x: pd.Timestamp(str(int(x))) if pd.notna(x) else pd.NaT
)

# ── 폐업 레이블 ────────────────────────────────────────────
# is_closed_obs: 관측기간(2023-2024) 내 폐업 (메인 레이블)
# is_closed_all: 전체 기간 폐업 포함 (2025 일괄처리 포함)
master['is_closed_obs'] = (
    master['MCT_ME_D_dt'].notna()
    & (master['MCT_ME_D_dt'] >= OBS_START)
    & (master['MCT_ME_D_dt'] <= OBS_END)
).astype(int)

master['is_closed_all'] = master['MCT_ME_D_dt'].notna().astype(int)

print(f"  is_closed_obs (2023-24 폐업): {master['is_closed_obs'].sum()}개")
print(f"  is_closed_all (전체 폐업):    {master['is_closed_all'].sum()}개")
print(f"  생존 점포:                    {(master['is_closed_obs']==0).sum()}개")
print()
print("업종 분포 (상위 10):")
print(master['HPSN_MCT_ZCD_NM'].value_counts().head(10))


---
## STEP 3. 월별 매출 데이터 정제 (dataset2)
버킷 컬럼(`'5_75-90%'` 형식)에서 앞의 숫자만 추출하여 정수형으로 변환.


In [ ]:
# ================================================================
# STEP 3: 월별 매출 데이터 정제 (dataset2)
# ================================================================

seongdong_ids = set(master['ENCODED_MCT'])
sales = df2[df2['ENCODED_MCT'].isin(seongdong_ids)].copy()
print(f" 매출 패널: {len(sales):,}행, {sales['ENCODED_MCT'].nunique():,}개 점포")

# ── 버킷 컬럼 파싱 ─────────────────────────────────────────
# 형식: '5_75-90%'  →  앞자리 숫자 5 (버킷 번호)
# 1=최상위(상위10%) ... 6=최하위(하위10%)
BUCKET_COLS_D2 = [
    'RC_M1_SAA',           # 매출액 구간
    'RC_M1_TO_UE_CT',      # 이용건수 구간
    'RC_M1_AV_NP_AT',      # 평균객단가 구간
    'M1_SME_RY_SAA_RAT',   # 업종 내 매출 순위 비율
    'M1_SME_RY_CNT_RAT',   # 업종 내 건수 순위 비율
    'MCT_UE_CLN_REU_RAT',  # 재방문율
    'RC_M1_SHC_FLP_UE_CLN_RAT',  # 유동인구 이용 비율
    'RC_M1_SHC_RSD_UE_CLN_RAT',  # 거주고객 이용 비율
]

for col in BUCKET_COLS_D2:
    if col in sales.columns:
        sales[col] = (
            sales[col].astype(str)
            .str.extract(r'^(\d+)')[0]
            .astype(float)
        )

# sentinel 처리
for col in sales.select_dtypes(include='number').columns:
    if col not in ('TA_YM', 'ENCODED_MCT'):
        sales[col] = sales[col].replace(SENTINEL, np.nan)
        sales[col] = sales[col].replace(-999999, np.nan)

# TA_YM 정수 정제 (202301.0 → 202301)
sales['TA_YM'] = pd.to_numeric(sales['TA_YM'], errors='coerce').astype('Int64')

print(f"\n버킷 컬럼 파싱 완료. 결측률:")
for col in BUCKET_COLS_D2[:4]:
    pct = sales[col].isna().mean() * 100
    print(f"  {col}: {pct:.1f}%")


---
## STEP 4. 월별 고객 데이터 정제 (dataset3)
sentinel 정수(-999999) 및 float(-999999.9) 처리 후 성동구 점포 필터링.


In [ ]:
# ================================================================
# STEP 4: 월별 고객 데이터 정제 (dataset3)
# ================================================================

cust = df3[df3['ENCODED_MCT'].isin(seongdong_ids)].copy()
print(f" 고객 패널: {len(cust):,}행, {cust['ENCODED_MCT'].nunique():,}개 점포")

# sentinel 교체: int/float 둘 다 처리
NUMERIC_COLS_D3 = [c for c in cust.select_dtypes(include='number').columns
                   if c not in ('TA_YM', 'ENCODED_MCT')]
SENTINEL_VALS = [SENTINEL, -999999]

for col in NUMERIC_COLS_D3:
    cust[col] = cust[col].replace(SENTINEL_VALS, np.nan)

# TA_YM 정수 정제
cust['TA_YM'] = pd.to_numeric(cust['TA_YM'], errors='coerce').astype('Int64')

print(f"\n고객 수치 컬럼 수: {len(NUMERIC_COLS_D3)}개")
print("결측률 상위 컬럼:")
miss = cust[NUMERIC_COLS_D3].isna().mean().sort_values(ascending=False)
print(miss.head(5).to_string())


---
## STEP 5. 패널 병합 + 파생 변수 생성
점포 마스터 × 월별 매출 × 월별 고객을 LEFT JOIN으로 병합.
`months_to_close`: 폐업 점포에 대해 마지막 관측월 기준 잔여 개월 계산.


In [ ]:
# ================================================================
# STEP 5: 패널 병합 + 파생 변수 생성
# ================================================================

# 매출 + 고객 병합
monthly = pd.merge(sales, cust, on=['ENCODED_MCT', 'TA_YM'], how='outer')
print(f"monthly (매출+고객): {monthly.shape}")

# 마스터 + monthly 병합
panel = pd.merge(master, monthly, on='ENCODED_MCT', how='left')
print(f"panel (전체):        {panel.shape}")

# ── TA_YM → datetime 변환 ──────────────────────────────────
panel['TA_YM_dt'] = panel['TA_YM'].apply(
    lambda x: pd.Timestamp(str(int(x)) + '01') if pd.notna(x) else pd.NaT
)

# ── 관측기간 필터 ──────────────────────────────────────────
panel = panel[
    panel['TA_YM_dt'].isna() | (
        (panel['TA_YM_dt'] >= OBS_START) &
        (panel['TA_YM_dt'] <= OBS_END)
    )
].copy()
print(f"관측기간 필터 후:   {panel.shape}")

# ── months_to_close (폐업까지 남은 개월) ─────────────────
def calc_months_to_close(row):
    if pd.isna(row['MCT_ME_D_dt']) or pd.isna(row['TA_YM_dt']):
        return np.nan
    diff = (row['MCT_ME_D_dt'].year - row['TA_YM_dt'].year) * 12 + \
           (row['MCT_ME_D_dt'].month - row['TA_YM_dt'].month)
    return max(diff, 0)

panel['months_to_close'] = panel.apply(calc_months_to_close, axis=1)

print(f"\n 패널 구성 완료")
print(f"   전체 점포:     {panel['ENCODED_MCT'].nunique():,}개")
print(f"   전체 행수:     {len(panel):,}행")
print(f"   폐업 점포(obs):{panel[panel['is_closed_obs']==1]['ENCODED_MCT'].nunique()}개")


---
## STEP 6. 검증 리포트 & 저장


In [ ]:
# ================================================================
# STEP 6: 검증 리포트 & 저장
# ================================================================
import os
os.makedirs(OUT_DIR, exist_ok=True)

print('=' * 60)
print(' 전처리 검증 리포트')
print('=' * 60)

n_stores  = panel['ENCODED_MCT'].nunique()
n_obs     = panel[panel['is_closed_obs']==1]['ENCODED_MCT'].nunique()
n_all     = panel[panel['is_closed_all']==1]['ENCODED_MCT'].nunique()
n_months  = panel['TA_YM'].nunique()
n_ind     = master['HPSN_MCT_ZCD_NM'].nunique()
n_dist    = master['HPSN_MCT_BZN_CD_NM'].nunique()

print(f"[1] 전체 점포 수:           {n_stores:,}개  (기대: ~4,168)")
print(f"[2] 관측 월수:              {n_months}개월  (기대: 24)")
print(f"[3] 폐업 점포 (관측기간):   {n_obs}개")
print(f"[4] 폐업 점포 (전체):       {n_all}개")
print(f"[5] 업종 수:               {n_ind}개")
print(f"[6] 상권 수:               {n_dist}개")
print(f"[7] 폐업율 (관측기간):      {n_obs/n_stores*100:.2f}%")
print()

# 저장
out_path = f'{OUT_DIR}panel_preprocessed.csv'
panel.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f" 저장 완료: {out_path}")
